# Alpha 因子评价

读取中性化 Alpha 暴露和 `return_t2`，在研究期非 ST 股票中计算每日 Rank IC，并逐因子汇总。2025 年作为回测期，不参与本步骤的因子筛选。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 文件路径与参数

自动向上定位项目根目录，使用项目相对路径读写文件；内核工作目录应位于本项目内。下方同时定义本步骤使用的参数。

评价信号日截止至 2024-12-27，其收益最晚于 2024-12-31 开盘实现；排除年末两个会用到 2025 年行情的信号日。


In [ ]:
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'environment.yml').is_file() and (path / 'src').is_dir()
)

EVA_START = pd.Timestamp('20230101')
# Last signal whose t+2 opening return is realized by 2024-12-31.
EVA_END = pd.Timestamp('20241227')

ALPHA = PROJECT_ROOT / "src/factor_exposure/alpha_factor/alpha_factor_exposure.parquet"
RETURN_T2 = PROJECT_ROOT / "src/return_t2/return_t2.parquet"
PANEL = PROJECT_ROOT / "data/processed/panel.parquet"

FIGURE_DIR = PROJECT_ROOT / 'src/figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 读取输入数据

Alpha 文件保存中性化后的 33 个暴露；面板提供 ST 状态，收益表提供评价标签。


In [ ]:
alpha = pd.read_parquet(ALPHA)
return_t2 = pd.read_parquet(RETURN_T2)
panel = pd.read_parquet(PANEL)

## 对齐评价样本

按日期和股票合并 ST 状态、收益标签及 Alpha 暴露，再筛选研究期内的非 ST 记录。每个因子的有效配对在计算相关性时单独确定。


In [ ]:
evaluation = pd.merge(left=panel[['ts_code', 'trade_date', 'ST']], right=return_t2, on=['trade_date','ts_code'], how='left')

In [ ]:
evaluation = pd.merge(left=evaluation, right=alpha, on=['trade_date','ts_code'], how='left')

In [ ]:
evaluation = evaluation[evaluation['trade_date'].between(EVA_START, EVA_END) & (evaluation['ST'] == False)]

## 预览样本

查看字段和筛选后的数据规模，确认日期、股票和标签已经对齐。


In [ ]:
evaluation.head()

In [ ]:
evaluation.shape

## 逐日计算 Rank IC

对每个交易日、每个 Alpha，计算暴露与 `return_t2` 的 Spearman 相关系数。相关性使用两列同时有效的记录，结果保存在 `rank_ic` 中。

正值表示高暴露股票的后续收益排序更高；负值表示相反。


In [ ]:
daily_ic = []

In [ ]:
for date, group in evaluation.groupby('trade_date'):
    for i in range(4, len(group.columns)):
        rank_ic = group.iloc[:,i].corr(
                group["return_t2"],
                method="spearman",
            )
        
        daily_ic.append({
            'alpha': group.columns[i],
            "trade_date": date,
            "rank_ic": rank_ic,
        })

In [ ]:
rank_ic = pd.DataFrame(daily_ic)

In [ ]:
rank_ic.head()

## 汇总评价结果

统计日均 IC、日 IC 样本标准差、ICIR 和正 IC 比例。

In [ ]:
evaluation_result = []

for alpha, group in rank_ic.groupby('alpha'):
    mean_ic = group['rank_ic'].mean()
    std_ic = group['rank_ic'].std(ddof=1)
    summarize = {
        'alpha': alpha,
        "mean_ic": mean_ic.round(4),
        "std_ic": std_ic.round(4),
        "icir": (mean_ic / std_ic).round(4),
        "positive_ic_ratio": (group['rank_ic'] > 0).mean().round(4)
        }
    evaluation_result.append(pd.Series(summarize))

In [ ]:
evaluation_result = pd.DataFrame(evaluation_result)

In [ ]:
evaluation_result

## Alpha 评价指标柱状图

In [ ]:
from matplotlib.ticker import PercentFormatter

metric_labels = {
    'mean_ic': 'Mean Rank IC',
    'std_ic': 'Rank IC Standard Deviation',
    'icir': 'Daily ICIR',
    'positive_ic_ratio': 'Positive Rank IC Ratio',
}

for metric, label in metric_labels.items():
    ordered = (
        evaluation_result[['alpha', metric]]
        .dropna(subset=[metric])
        .sort_values(metric, ascending=False, kind='stable')
    )
    colors = np.where(ordered[metric] >= 0, '#2878B5', '#D9534F')
    fig, ax = plt.subplots(figsize=(16, 6), layout='constrained')
    ax.bar(ordered['alpha'], ordered[metric], color=colors, width=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'Alpha Factors: {label} (Descending)')
    ax.set_xlabel('Alpha Factor')
    ax.set_ylabel(label)
    ax.tick_params(axis='x', labelrotation=60, labelsize=9)
    plt.setp(ax.get_xticklabels(), ha='right')
    ax.set_axisbelow(True)
    ax.grid(axis='y', alpha=0.25)
    if metric == 'positive_ic_ratio':
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    fig.savefig(FIGURE_DIR / f'evaluation_{metric}.png', dpi=200)
    plt.show()
    plt.close(fig)

## 初步候选因子的研究价值

仅依据上述四项评价指标，初步保留 **Alpha42、Alpha80、Alpha1、Alpha132、Alpha95、Alpha112、Alpha63、Alpha31、Alpha177、Alpha110、Alpha118** 共 11 个因子，作为后续研究候选。此名单不沿用已有收益模型的因子选择。

### 初筛依据

采用探索性门槛 `|mean_ic| ≥ 0.01`、`|icir| ≥ 0.1`，并检查正 IC 比例是否支持平均 IC 的方向。门槛用于整理当前研究期的候选，不是普遍有效的标准。负 IC 表示反向排序关系，不能直接视为无效；后续可对暴露取负，统一研究方向。

### 各因子的研究价值

| 因子 | mean_ic | std_ic | icir | 正 IC 比例 | 研究方向 | 研究价值与验证重点 |
| --- | ---: | ---: | ---: | ---: | --- | --- |
| Alpha42 | 0.0145 | 0.0523 | 0.2764 | 59.96% | 正向 | 正向 ICIR 最高，IC 波动相对较低，方向一致性较好；优先检验其跨阶段持续性。 |
| Alpha80 | -0.0190 | 0.0674 | -0.2823 | 38.59% | 反向 | 绝对 ICIR 最高，较强的负向平均 IC 与较低的正 IC 比例相互支持；优先研究反向使用后的表现。 |
| Alpha1 | 0.0104 | 0.0528 | 0.1966 | 56.02% | 正向 | 平均 IC 中等，但波动较低，适合作为相对稳定的正向信号候选。 |
| Alpha132 | 0.0189 | 0.0993 | 0.1899 | 58.51% | 正向 | 正向平均 IC 最高，方向频率也较好，但日度波动较大；关注强度是否集中于少数阶段。 |
| Alpha95 | 0.0160 | 0.0859 | 0.1861 | 56.02% | 正向 | 正向平均强度和 ICIR 均较好，可与 Alpha1、Alpha132 比较强度与稳定性的取舍。 |
| Alpha112 | -0.0211 | 0.1052 | -0.2005 | 42.53% | 反向 | 负向平均 IC 较强，绝对 ICIR 居前，但波动偏高；需检查极端日期及阶段依赖。 |
| Alpha63 | -0.0214 | 0.1122 | -0.1908 | 43.57% | 反向 | 平均 IC 绝对值最高，但波动也较高；适合检验较强的平均关系能否在不同阶段延续。 |
| Alpha31 | -0.0209 | 0.1123 | -0.1858 | 43.78% | 反向 | 负向平均关系较强，但 IC 波动较高；关注其与其他强负向因子的差异及信息重叠。 |
| Alpha177 | -0.0153 | 0.0820 | -0.1865 | 44.19% | 反向 | 平均负向强度与相对稳定性较均衡；可与高强度、高波动的反向候选进行比较。 |
| Alpha110 | -0.0160 | 0.0895 | -0.1785 | 45.23% | 反向 | 平均 IC 和绝对 ICIR 达到初筛门槛，但方向频率优势较弱；需重点验证持续性。 |
| Alpha118 | -0.0145 | 0.0818 | -0.1775 | 43.57% | 反向 | 负向关系和方向频率支持保留研究，但绝对 ICIR 接近门槛；需验证独立信息及增量价值。 |


## 候选因子的平均截面相关矩阵

逐交易日计算股票截面上的 Spearman 相关矩阵，每对因子仅使用共同有效的股票（至少 3 只）；样本不足或暴露为常数时相关系数为空。对每个矩阵元素按有效交易日等权取算术平均，不将缺失相关系数填零。图中 `(-)` 表示原始暴露已取负。

图表保存至 `src/figures/evaluation_selected_11_mean_correlation.png`。

In [ ]:
selected_alphas = [
    'alpha42', 'alpha80', 'alpha1', 'alpha132', 'alpha95',
    'alpha112', 'alpha63', 'alpha31', 'alpha177', 'alpha110', 'alpha118',
]
negative_alphas = [
    'alpha80', 'alpha112', 'alpha63', 'alpha31',
    'alpha177', 'alpha110', 'alpha118',
]

# Align directions on a copy without changing the evaluation data.
selected_exposures = evaluation[['trade_date', *selected_alphas]].copy()
selected_exposures[negative_alphas] = -selected_exposures[negative_alphas]

# Pairwise-complete cross-sectional correlations, followed by equal-day means.
daily_alpha_correlations = pd.concat(
    {
        date: group[selected_alphas].corr(method='spearman', min_periods=3)
        for date, group in selected_exposures.groupby('trade_date')
    },
    names=['trade_date', 'alpha'],
)
mean_alpha_correlation = (
    daily_alpha_correlations.groupby(level='alpha').mean()
    .reindex(index=selected_alphas, columns=selected_alphas)
)

labels = [
    f'{name} (-)' if name in negative_alphas else name
    for name in selected_alphas
]
fig, ax = plt.subplots(figsize=(12, 10), layout='constrained')
heatmap = ax.imshow(
    np.ma.masked_invalid(mean_alpha_correlation.to_numpy()),
    cmap='RdBu_r', vmin=-1, vmax=1,
)
ax.set_xticks(range(len(labels)), labels=labels, rotation=60, ha='right')
ax.set_yticks(range(len(labels)), labels=labels)
ax.set_title('Selected 11 Alphas: Mean Daily Spearman Correlation\nDirection-Aligned Exposures')
for row in range(len(labels)):
    for col in range(len(labels)):
        value = mean_alpha_correlation.iloc[row, col]
        ax.text(
            col, row, f'{value:.2f}' if pd.notna(value) else 'N/A',
            ha='center', va='center', fontsize=9,
            color='white' if pd.notna(value) and abs(value) > 0.5 else 'black',
        )
fig.colorbar(heatmap, ax=ax, label='Mean Daily Spearman Correlation', shrink=0.85)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_DIR / 'evaluation_selected_11_mean_correlation.png', dpi=200)
plt.show()
plt.close(fig)

## 后续研究因子的保留依据

结合研究期内的单因子评价结果和方向统一后的平均日截面 Spearman 相关矩阵，暂时保留 **Alpha42、Alpha80、Alpha1、Alpha132、Alpha112、Alpha177、Alpha118**，作为后续研究的七因子基准集合。其中 **Alpha80、Alpha112、Alpha177、Alpha118 使用取负后的暴露**，其余保持原方向。以下 IC 数值仍按原始方向列示，相关系数则采用方向统一后的口径。

选择原则是兼顾预测强度、IC 相对稳定性与因子间的信息重叠。相关性用于辅助缩减候选，不作为证明因子冗余或组合有效性的唯一依据。

### 保留七个因子的理由

| 因子 | 单因子表现 | 与其他候选的关系及保留理由 |
| --- | --- | --- |
| Alpha42 | mean_ic = 0.0145，icir = 0.2764，正 IC 比例为 59.96% | 正向 ICIR 最高，IC 波动相对较低；与 Alpha80 的平均相关系数仅为 0.03，与其余保留因子的相关性也较低，优先保留。 |
| Alpha80 | mean_ic = -0.0190，icir = -0.2823，正 IC 比例为 38.59% | 绝对 ICIR 在初选因子中最高，支持反向使用；与其他保留因子的平均相关性较低，兼具单因子强度和较少的排序重叠。 |
| Alpha1 | mean_ic = 0.0104，std_ic = 0.0528，icir = 0.1966 | 平均 IC 中等，但波动较小；与 Alpha42、Alpha80 的相关系数分别为 0.16、0.07，与 Alpha132 为 0.02，适合保留作相对稳定的正向信号。 |
| Alpha132 | mean_ic = 0.0189，icir = 0.1899，正 IC 比例为 58.51% | 与 Alpha95 高度相关（0.86），两者暂选其一。Alpha132 的平均 IC、ICIR 和正 IC 比例均略优，且与其余保留因子的平均相关性接近零，因此优先保留。 |
| Alpha112 | mean_ic = -0.0211，std_ic = 0.1052，icir = -0.2005 | 与 Alpha63、Alpha31 的相关系数分别为 0.83、0.75，三者存在明显重叠。Alpha112 的绝对 ICIR 更高、IC 标准差更低，暂作为该组的代表。 |
| Alpha177 | mean_ic = -0.0153，icir = -0.1865，正 IC 比例为 44.19% | 与 Alpha112 的相关系数为 0.51，存在一定重叠，但与 Alpha42、Alpha80、Alpha1、Alpha132 的相关系数分别为 0.24、0.10、0.16、-0.01，暂保留以检验增量价值。 |
| Alpha118 | mean_ic = -0.0145，icir = -0.1775，正 IC 比例为 43.57% | 与 Alpha110 的相关系数为 0.69，两者绝对 ICIR 非常接近。Alpha118 与 Alpha112、Alpha177 的相关系数为 0.48、0.44，低于 Alpha110 对应的 0.58、0.52，因此暂优先保留 Alpha118，以减少与已选因子的重叠。 |

### 其余四个因子的处理

- **Alpha95：作为 Alpha132 的替代候选。** 两者平均相关系数达 0.86，暂不同时纳入基准集合；后续可比较 Alpha95 较低的 IC 波动是否能改善组合表现。
- **Alpha63、Alpha31：作为 Alpha112 的替代候选。** 两者之间的相关系数为 0.90，与 Alpha112 也高度相关；当前优先采用绝对 ICIR 更高的 Alpha112，但不据此断定另外两个没有增量信息。
- **Alpha110：作为 Alpha118 的替代候选。** 两者的绝对 ICIR 差异很小，当前主要依据与其余已选因子的重叠程度取舍。这项选择的确定性相对较低，应优先进行替换比较。